In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from dotenv import load_dotenv

load_dotenv()

from myapp import create_app

app = create_app()
app.app_context().push()

In [2]:
# test koneksi
# test koneksi ke database
from sqlalchemy import text
from myapp.extensions import db
from sqlalchemy import select

result = db.session.execute(text("SELECT 1"))

print(result.scalar())

1


In [ ]:
from myapp.extensions import db
from sqlalchemy import select

# service
from myapp.features.auth.services import authenticate

# repository
from myapp.features.auth.repository import user_repository

# models
from myapp.features.auth.models import User
from myapp.features.surat.models import (
    SuratKeluar, SuratMasuk
)

# utils
from myapp.features.auth.utils import *

from werkzeug.security import check_password_hash

surat repository

In [10]:
from myapp.features.surat import repository

surat_keluar_repo = repository.SuratKeluarRepository()
surat_masuk_repo = repository.SuratMasukRepository()

In [ ]:
# surat keluar
stmt = select(SuratKeluar).limit(5)
surat_keluar_list = db.session.scalars(stmt).all()
surat_keluar_list

In [ ]:
surat_keluar = surat_keluar_list[0]
surat_keluar.keterangan
surat_keluar.isi_singkat

repository

In [2]:
from myapp.features.surat import repository

surat_masuk_repo = repository.SuratMasukRepository()
surat_keluar_repo = repository.SuratKeluarRepository()
# surat_masuk_repo.get_all()
# surat_masuk_repo.get(2)
surat_masuk_repo.filter_by(id=1)

[<SuratMasuk 1>]

pencarian

In [6]:
from myapp.features.mesin_pencarian.services import (
    SearchEngine, SearchEngineSurat, SearchEngine3
)





In [13]:
mesin_pencarian = SearchEngine3(surat_keluar_repo.get_all())
mesin_pencarian.search("Surat ")

[(<SuratKeluar 4>, np.float64(0.46233742855496285)),
 (<SuratKeluar 26>, np.float64(0.46233742855496285)),
 (<SuratKeluar 32>, np.float64(0.46233742855496285)),
 (<SuratKeluar 15>, np.float64(0.4420229957796241)),
 (<SuratKeluar 25>, np.float64(0.4420229957796241))]

In [ ]:

engine = SearchEngine3(surat_keluar_repo.get_all())
# engine = SearchEngineSurat(surat_keluar_repo.get_all())
query = "Surat444    contoh"
hasil = engine.search(query)

for surat, score in hasil:
    print(surat.id, end=" ")
    print(surat.nomor_surat, end=" ")
    print(surat.isi_singkat, end=" ")
    print(round(score, 3))

2 450.2/01/I/2026 Perangkat nikah 0.0
4 511.1/02/I/2026 Surat keterangan usaha 0.0
6 511.1/10/I/2026 Keterangan usaha 0.0
7 474.1/12/I/2026 Formulir KK baru 0.0
8 404/17/I/2026 Keterangan tidak mampu 0.0


In [ ]:
surat_list = surat_masuk_repo.get_all()

documents = [
    surat.isi_singkat
    for surat in surat_list
    if surat.isi_singkat
]

engine = SearchEngine(documents)

In [ ]:
hasil = engine.search("python flask")
for surat, score in hasil:
    print(surat, score)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

documents = [
    "Belajar Python untuk pemula",
    "Tutorial Flask menggunakan Python",
    "Belajar Machine Learning dengan Python",
    "Cara memasak nasi goreng"
]

query = "python flask"

# Menggabungkan query dan dokumen
all_text = [query] + documents

# Mengubah menjadi TF-IDF
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(all_text)

# Query adalah baris pertama
query_vector = tfidf_matrix[0]

# Dokumen adalah sisanya
document_vectors = tfidf_matrix[1:]

# Hitung cosine similarity
scores = cosine_similarity(query_vector, document_vectors)

for doc, score in zip(documents, scores[0]):
    print(f"{score:.3f} -> {doc}")

urutkan

In [ ]:
results = list(zip(documents, scores[0]))

results.sort(key=lambda x: x[1], reverse=True)

for doc, score in results:
    print(f"{score:.3f} -> {doc}")

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


class SearchEngine:

    def __init__(self, documents):
        self.documents = documents
        self.vectorizer = TfidfVectorizer()

        self.document_vectors = self.vectorizer.fit_transform(documents)

    def search(self, query, top_k=5):
        query_vector = self.vectorizer.transform([query])

        scores = cosine_similarity(
            query_vector,
            self.document_vectors
        )[0]

        results = list(zip(self.documents, scores))

        results.sort(key=lambda x: x[1], reverse=True)

        return results[:top_k]


documents = [
    "Belajar Python untuk pemula",
    "Tutorial Flask menggunakan Python",
    "Belajar Machine Learning dengan Python",
    "Cara memasak nasi goreng"
]

engine = SearchEngine(documents)

hasil = engine.search("python flask")

for doc, score in hasil:
    print(score, doc)

In [ ]:
# products = session.query(Product).all()

# documents = [p.name for p in products]

# engine = SearchEngine(documents)

# hasil = engine.search("laptop asus")